# Imports

In [ ]:
%%capture
%pip install "transformers>=5.0.0"

In [ ]:
import torch
import os
from transformers import AutoProcessor, AutoModelForMultimodalLM, AutoModelForImageTextToText
import numpy as np
import cv2
import datetime as dt
import json, re

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32 if device == "cpu" else torch.bfloat16

# Params

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    is_colab = True
except ImportError:
    print("Google Colab environment not detected.")
    is_colab = False

In [ ]:
DATASET_NAME = "ICDAR03/lfsosa"
DATASET_PATH = f"output/{DATASET_NAME}/" if not is_colab else f"/content/drive/MyDrive/Thesis/resources/{DATASET_NAME}/"
BATCH_SIZE = 15

# Model Loading

In [ ]:
company_name = "lightonai"
model_name = "LightOnOCR-2-1B"

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN", None)
NBR_TOKENS = 256 if model_name == "LightOnOCR-2-1B" else 384 

## (Down-)load the model and processor

In [ ]:
# if the Models/{model_name} dir exists, we load from it. Other we create it and save the model there
if os.path.exists(f"Models/{model_name}"):
    print(f"Loading model {model_name} from Models/{model_name}")
    processor = AutoProcessor.from_pretrained(f"Models/{model_name}", use_auth_token=HF_TOKEN)
    model = AutoModelForMultimodalLM.from_pretrained(f"Models/{model_name}").to(device)
else:
    print(f"Creating model {model_name} in Models/{model_name}")
    processor = AutoProcessor.from_pretrained(company_name + "/" + model_name, use_auth_token=HF_TOKEN, trust_remote_code=True)
    model = AutoModelForMultimodalLM.from_pretrained(company_name + "/" + model_name, trust_remote_code=True).to(device)
    processor.save_pretrained(f"Models/{model_name}")
    model.save_pretrained(f"Models/{model_name}")

In [ ]:
print(f"Model is on device: {model.device}")

# Model Inference

In [ ]:
import re
from bs4 import BeautifulSoup

def extract_content(string:str, model_name:str):
    """Short function to extract the content from the string based on the model name."""
    match model_name:
        case "LightOnOCR-2-1B":
            results = light_on_ocr_to_word_list(string)
            return results
        case "gemma-3-4b-it":
            # For gemma-3-4b-it, we need to extract the JSON content
            return gemma_3_to_word_list(string)
        case "PaddleOCR-VL-1.6":
            return string
        case "NuExtract3":
            return string
        case _:
            return string
    
def natural_sort_key(value):
    parts = re.split(r'(\d+)', value)
    return [int(part) if part.isdigit() else part.lower() for part in parts]

def save_predictions_checkpoint(path, data):
    temp_path = f"{path}.tmp"
    with open(temp_path, "w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2, ensure_ascii=False)
    os.replace(temp_path, path)

def gemma_3_to_word_list(text: str) -> list[str]:
    words = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue  # on ignore les lignes vides
        words.extend(line.split())  # découpe la ligne en mots sur les espaces
    return words

def light_on_ocr_to_word_list(text: str) -> list[str]:
    # We remove all the text that is after the "Note:" line, as it is not part of the transcription
    img_index = text.lower().find("![image]")
    if img_index != -1:
        text = text[:img_index]
    note_index = text.lower().find("Note:")
    if note_index != -1:
        text = text[:note_index]
    
    """soup = BeautifulSoup(text, "html.parser")
    text_no_html = soup.get_text(separator=" ")

    text_clean = re.sub(r'\|', ' ', text_no_html)
    text_clean = re.sub(r'\$.*?\$', '', text_clean)  # Remove math mode
    text_clean = re.sub(r'(?im)^\s*Note:\s*.*$', '', text_clean)
    text_clean = re.sub(r'^[-:=]{3,}$', '', text_clean, flags=re.MULTILINE)
    text_clean = re.sub(r'#{1,6}\s*', '', text_clean)
    text_clean = re.sub(r'\*{1,2}([^*]+)\*{1,2}', r'\1', text_clean)
    text_clean = re.sub(r'^\s*[-*+]\s+', '', text_clean, flags=re.MULTILINE)

    words = text_clean.strip().split()"""

    return text

In [ ]:
match model_name:
    case "gemma-3-4b-it":
        instructions = [
            "You are an OCR engine. You do not chat, explain, or comment.",
            "Locate all text in the image and transcribe it exactly as it appears.",
            "Output ONLY the raw transcribed text. Nothing else.",
            "Do NOT add any introduction (e.g. 'Here is the text...').",
            "Do NOT add any conclusion, offer, or question (e.g. 'Let me know if...').",
            "Do NOT use Markdown formatting: no bullet points, no bold (**), no headers (#), no asterisks.",
            "Do NOT interpret, label, or reformat the content — just transcribe what is visually present, line by line.",
            "Your entire response must be the transcription and nothing else.",
        ]
    case "LightOnOCR-2-1B":
        # Never use instructions for LightOnOCR-2-1B, as it is a vision model and does not require any instructions. The model will output the text directly.
        instructions = []
    case _:
        instructions = [
            "You are an OCR engine. You do not chat, explain, or comment.",
            "Locate all text in the image and transcribe it exactly as it appears.",
            "Output ONLY the raw transcribed text. Nothing else.",
            "Do NOT add any introduction (e.g. 'Here is the text...').",
            "Do NOT add any conclusion, offer, or question (e.g. 'Let me know if...').",
            "Do NOT use Markdown formatting: no bullet points, no bold (**), no headers (#), no asterisks.",
            "Do NOT interpret, label, or reformat the content — just transcribe what is visually present, line by line.",
            "Your entire response must be the transcription and nothing else.",
        ]
        

CHECKPOINT_PATH = os.path.join("resources/checkpoints/", f"{DATASET_NAME.replace('/', '_')}_{model_name}_predictions.json")

In [ ]:
# We try on a single image first to see if the model is working properly
file = "IMG_2469.JPG"
image = cv2.imread(os.path.join(DATASET_PATH, file))
image = cv2.resize(image, (1024, 1024), interpolation=cv2.INTER_CUBIC)

messages = [{"role": "user","content": [{"type": "image", "image": image},{"type": "text", "text": "".join(instructions)}],}]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=NBR_TOKENS)
generated_ids = outputs[0, inputs["input_ids"].shape[1]:]
result = processor.decode(generated_ids, skip_special_tokens=True)
print(f"Raw prediction for {file}: {result}")
clean_result = extract_content(result, model_name)
print(f"Prediction for {file}: {clean_result}")

In [ ]:
predictions = {}
back_up_predictions = {}

image_files = sorted(
    [file for file in os.listdir(DATASET_PATH) if file.lower().endswith((".jpg", ".jpeg", ".png"))],
    key=lambda file: natural_sort_key(os.path.splitext(file)[0])
)

CHECKPOINT_PATH = os.path.join("resources/checkpoints/", f"{DATASET_NAME.replace('/', '_')}_{model_name}_predictions.json")

if os.path.exists(CHECKPOINT_PATH) and os.path.getsize(CHECKPOINT_PATH) > 0:
    try:
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as handle:
            predictions = json.load(handle)
        print(f"Resuming from checkpoint with {len(predictions)} processed images: {CHECKPOINT_PATH}")
    except json.JSONDecodeError:
        print(f"Checkpoint file could not be read, starting from scratch: {CHECKPOINT_PATH}")

start_time = dt.datetime.now()
last_checkpoint_time = start_time

print(f"Processing {len(image_files)} images in {DATASET_PATH}...")

for batch_start in range(0, len(image_files), BATCH_SIZE):
    batch_files = image_files[batch_start:batch_start + BATCH_SIZE]
    batch_names = [os.path.splitext(file)[0] for file in batch_files]
    
    if not batch_names:
        continue  # Skip empty batches
    
    print(f"Processing batch [{batch_names[0]} to {batch_names[-1]}]")
    
    for file in batch_files:
        if file.endswith((".jpg", ".jpeg", ".png", '.PNG', '.JPG', '.JPEG')):
            image_path = os.path.join(DATASET_PATH, file)
            image_name = os.path.splitext(file)[0]
            
            if image_name in predictions:
                continue
            
            image = cv2.imread(image_path)
            image = cv2.resize(image, (1024, 1024), interpolation=cv2.INTER_CUBIC)
            
            messages = [{"role": "user","content": [{"type": "image", "image": image},{"type": "text", "text": "".join(instructions)},]}]
            
            inputs = processor.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            ).to(model.device)
            
            outputs = model.generate(**inputs, max_new_tokens=NBR_TOKENS)
            generated_ids = outputs[0, inputs["input_ids"].shape[1]:]
            result = processor.decode(generated_ids, skip_special_tokens=True)
            clean_result = extract_content(result, model_name)
            back_up_predictions[image_name] = result
            predictions[image_name] = clean_result
            
    step_time = dt.datetime.now()
    elapsed_time = step_time - start_time
    minutes, seconds = divmod(elapsed_time.total_seconds(), 60)
    last_checkpoint_time = step_time
    
    save_predictions_checkpoint(CHECKPOINT_PATH, predictions)
    print(f"Checkpoint saved after batch [{batch_names[0]} to {batch_names[-1]}]: {len(predictions)} images in {int(minutes)} minutes and {int(seconds)} seconds.")

In [ ]:
# Format the predictions dictionary as a JSON object
pred = {}
DEBUG = False
for img_name, text in predictions.items():
    # If the predicted text is empty, store an empty dictionary.
    # Else if the predicted text is a string, we split it into lines and store it as a list of strings.
    # Else if the predicted text is a list, we split each string in the list into lines and store into a unique list of strings.
    words = []
    if text == "":
        print(f"No text found in {img_name}. Storing an empty dictionary.") if DEBUG else None
        pred[img_name] = {}
    elif isinstance(text, str):
        print(f"Text found in {img_name}. Splitting into lines and storing as a list of strings.") if DEBUG else None
        pred[img_name] = text.split()
    elif isinstance(text, list):
        print(f"Text found in {img_name}. Splitting each string in the list into lines and storing into a unique list of strings.") if DEBUG else None
        for l in text:
            print(f"\tProcessing list item: {l}") if DEBUG else None
            if isinstance(l, str):
                print(f"\t\tSplitting string into lines: {l}") if DEBUG else None
                words.extend(l.split())
            elif isinstance(l, list):
                print(f"\t\tProcessing nested list: {l}") if DEBUG else None
                words.extend([line for line in l if isinstance(line, str) and line.strip()])
        pred[img_name] = words
    else:
        print(f"Unexpected data type for {img_name}: {type(text)}. Storing an empty dictionary.") if DEBUG else None
        pred[img_name] = {}

# Save the predictions to a JSON file
print(f"Saving predictions to output/predicitions/{DATASET_NAME}_{model_name}.json")
with open(f"output/predictions/{DATASET_NAME}_{model_name}.json", "w") as f:
    json.dump(pred, f, indent=2, ensure_ascii=False)